# TrustLung AI — Explainability & Uncertainty

This notebook covers:
- Grad-CAM heatmap generation
- Monte Carlo Dropout uncertainty estimation
- Reliability (calibration) diagram
- False positive reduction demonstration

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.utils.helpers import load_config, set_seed
from src.preprocessing.data_loader import generate_synthetic_dataset
from src.models.architectures import get_model
from src.models.uncertainty import MCDropoutPredictor, plot_uncertainty_distribution, plot_confidence_vs_accuracy, plot_mc_samples
from src.models.fp_reduction import apply_confidence_threshold, ProbabilityCalibrator, false_positive_analysis
from src.explainability.gradcam import GradCAMVisualizer

set_seed(42)
config = load_config('../configs/config.yaml')
data   = generate_synthetic_dataset(n_samples=300, seed=42)
print('Data ready.')

## 1. Build a Quick Demo Model

In [ ]:
# Build EfficientNetB0 (pretrained=False for fast demo)
model = get_model('EfficientNetB0', input_shape=(224,224,3), num_classes=3,
                   dropout_rate=0.4, pretrained=True)
model.summary()

## 2. Monte Carlo Dropout Uncertainty

In [ ]:
mc_pred = MCDropoutPredictor(model, n_samples=30, class_names=data['class_names'])

# Predict first 50 test images
results  = mc_pred.predict_batch_with_uncertainty(data['X_test'][:50])
agg      = mc_pred.aggregate_batch_results(results)

print('Sample predictions:')
for i in range(5):
    r = results[i]
    print(f"  [{i}] Pred={r['predicted_label']:10s} Conf={r['confidence']:.1%} "
          f"Unc={r['uncertainty']:.3f} ({r['uncertainty_label']})")

In [ ]:
# Plot MC sample distribution for one image
plot_mc_samples(
    results[0]['all_samples'],
    class_names=data['class_names'],
    title='MC Dropout Samples — Sample #0'
)
plt.show()

In [ ]:
# Uncertainty distribution across classes
plot_uncertainty_distribution(
    agg['uncertainty'], data['y_test_raw'][:50],
    class_names=data['class_names'],
    save_path='../outputs/plots/uncertainty_dist_notebook.png'
)
plt.show()

## 3. Calibration (Reliability Diagram)

In [ ]:
plot_confidence_vs_accuracy(
    agg['confidence'], agg['y_pred'], data['y_test_raw'][:50],
    n_bins=10,
    save_path='../outputs/plots/reliability_notebook.png'
)
plt.show()

## 4. False Positive Reduction

In [ ]:
y_pred_raw     = agg['y_pred']
y_pred_reduced = apply_confidence_threshold(agg['y_prob'], threshold=0.75)

fp_report = false_positive_analysis(
    y_true=data['y_test_raw'][:50],
    y_pred_before=y_pred_raw,
    y_pred_after=y_pred_reduced,
    class_names=data['class_names'],
    save_path='../outputs/plots/fp_reduction_notebook.png'
)
plt.show()
print('\nFP Reduction Summary:')
for phase, metrics in fp_report.items():
    print(f'  {phase}:', {k: f"{v:.4f}" for k,v in metrics.items()})

## 5. Grad-CAM Explanation Grid

In [ ]:
visualizer = GradCAMVisualizer(
    model=model,
    class_names=data['class_names'],
    output_dir='../outputs/heatmaps/notebook_demo'
)

# Explain first 8 test images
visualizer.explain_batch(
    data['X_test'][:8],
    data['y_test_raw'][:8],
    max_samples=8
)

# Grid visualization
visualizer.explanation_grid(
    data['X_test'][:8], data['y_test_raw'][:8],
    n_cols=4,
    save_path='../outputs/plots/gradcam_grid_notebook.png'
)
plt.show()